# RL 모듈 — 최종 검증 실행 (Colab)

지금까지 찾은 버그 3개(거래비용 2배 계산, 무위험이자율 하드코딩, MACD look-ahead bias)와 VecNormalize가 **전부 한꺼번에 적용된 상태로 Walk-Forward 검증을 다시 돌리는 노트북**입니다.

**실행 방법**: 위 메뉴 `런타임 > 모두 실행` (GPU 필요 없음)

**예상 소요시간**: 약 25~35분 (셀 3번, 시드 2개 기준)

**각 셀은 실패하면 바로 에러를 띄우고 멈추도록 만들어뒀습니다** — 셀 2나 3에서 멈추면, 그 셀의 출력(위쪽 스크롤)에 실제 에러 메시지가 있습니다.

In [ ]:
# 셀 1: 레포 클론 + 의존성 설치 (재실행해도 안전하도록 처리)
import os
if os.path.exists('robo-advisor-rl'):
    %cd robo-advisor-rl
    !git pull --quiet
else:
    !git clone --quiet https://github.com/kimcanal/robo-advisor-rl.git
    %cd robo-advisor-rl
!pip install -q -r rl/requirements.txt
print("\n셀 1 완료")

In [ ]:
# 셀 2: 실제 ETF 10종 데이터 받기
!python -m rl.data.fetch_real_data
assert _exit_code == 0, (
    f"fetch_real_data.py가 실패했습니다 (exit code {_exit_code}). "
    "위 출력에서 에러 메시지를 확인하세요 — Yahoo Finance가 Colab IP를 "
    "일시적으로 막았을 수 있습니다 (잠시 후 이 셀만 재실행해보세요)."
)

# 실제로 파일이 제대로(비어있지 않게) 받아졌는지 재확인
tickers = ["SPY", "QQQ", "IWM", "EFA", "EEM", "AGG", "TLT", "HYG", "GLD", "VNQ"]
missing = [t for t in tickers if not os.path.exists(f"rl/data/raw/{t}.csv") or os.path.getsize(f"rl/data/raw/{t}.csv") < 100]
assert not missing, f"다음 티커 데이터가 비어있거나 없습니다: {missing} — 이 셀을 다시 실행해보세요."
print(f"\n셀 2 완료 — {len(tickers)}개 티커 데이터 확인됨")

In [ ]:
# 셀 3: 최종 Walk-Forward 검증 (2개 윈도우 x 보상함수 3종 x 시드 2개, 전체 버그 수정 반영)
# 약 25~35분 소요
!python -m rl.walk_forward --real --timesteps 120000 --seeds 0,1
assert _exit_code == 0, (
    f"walk_forward.py가 실패했습니다 (exit code {_exit_code}). "
    "위 출력에서 에러 메시지(Traceback)를 찾아 확인하세요."
)
print("\n셀 3 완료")

In [ ]:
# 셀 4: 결과 확인
import pandas as pd

results = pd.read_csv('rl/outputs/walk_forward/walk_forward_results.csv')
print('=== 윈도우별 요약 ===')
display(results[['window', 'strategy', 'seed', 'cumulative_return', 'sharpe', 'mdd']])

daily = pd.read_csv('rl/outputs/walk_forward/walk_forward_daily.csv')
print(f"\n일별 레코드 수: {len(daily)}")
print(daily['regime'].value_counts())

In [ ]:
# 셀 5 (선택): 결과 CSV를 로컬로 다운로드
from google.colab import files
files.download('rl/outputs/walk_forward/walk_forward_results.csv')
files.download('rl/outputs/walk_forward/walk_forward_daily.csv')